# File 1 — Resume Preprocessing, Security & Embedding Pipeline

## Advanced Resume Optimization + RAG System

**Stage:** Preprocessing · PII Sanitization · Structured Schema Extraction · Embedding Generation  
**File:** 1 of 2  
**Output Files:** `structured_resume.json` · `pii_vault.json` · `resume_embeddings.npy`

---

## What This Notebook Does

This notebook is the **first stage** of a two-file RAG pipeline. It handles everything upstream of generation and retrieval:

| Step | Description |
|------|-------------|
| 1 | Extract raw text from a resume (PDF or DOCX) |
| 2 | Parse resume into a typed JSON schema |
| 3 | Detect and sanitize all PII using deterministic SHA-256 tokens |
| 4 | Classify and tokenize URLs/profiles (GitHub, LinkedIn, etc.) |
| 5 | Embed the sanitized resume using `BAAI/bge-base-en-v1.5` |
| 6 | Persist outputs to disk for consumption by File 2 |

**File 2** will handle ChromaDB ingestion, Top-K retrieval, job description matching, critic loop, and ATS scoring.  
**Do NOT** add ChromaDB queries or generation logic to this notebook.

---

## Security & Privacy Notes

> ⚠️ **READ BEFORE RUNNING**

- **Salt management:** The PII hashing salt is **never hardcoded** in this notebook. It must be provided at runtime via the `PII_SALT` environment variable or a secure prompt. Store the salt in a secrets manager or `.env` file that is excluded from version control.
- **Vault protection:** `pii_vault.json` contains the mapping from tokens to original PII values. Restrict permissions immediately after creation:
  ```bash
  chmod 600 pii_vault.json   # Linux/macOS
  icacls pii_vault.json /inheritance:r /grant:r "%USERNAME%:F"  # Windows
  ```
- **Irreversible hashing:** SHA-256 is used as a one-way function. Restoration requires the vault file — the hash alone cannot be reversed.
- **No plaintext PII in logs:** All logging is masked. Original values are never written to stdout, stderr, or log files.
- **Restoration authorization:** The `restore_pii()` function requires explicit vault access. Treat vault access as a privileged operation. Audit all calls in production.
- **Vault at rest:** In production, encrypt `pii_vault.json` at rest (e.g., AES-256 via `cryptography` library) before committing to persistent storage.

---

## Section 1 — System Configuration

All pipeline parameters are centralized in the `CONFIG` dictionary. Modify values here to adjust chunking, model selection, or PII detection scope without touching the pipeline code.

In [2]:
# ────────────────────────────────────────────────────────────────
# SYSTEM CONFIGURATION
# Centralised config — modify here, not in the pipeline functions.
# ────────────────────────────────────────────────────────────────

CONFIG = {
    # HuggingFace sentence-transformer model for dense embeddings
    "embedding_model": "BAAI/bge-base-en-v1.5",

    # Text chunking parameters (used for optional chunk-level embeddings)
    "chunk_size": 300,          # tokens per chunk
    "chunk_overlap": 50,        # overlap between consecutive chunks

    # PII detection categories — all active by default
    "pii_fields": [
        "name",             # full names, initials, suffixes
        "phone",            # all country/format variants
        "email",            # standard + obfuscated forms
        "address",          # street, city, postal codes
        "government_id",    # Aadhaar, SSN, PAN, Passport, etc.
        "demographics",     # DOB, gender, nationality, religion hints
        "financial",        # bank accounts, salary figures, tax info
        "urls",             # http(s)://, www., shorteners
        "profiles",         # github, gitlab, linkedin, stackoverflow
        "social_handles",   # @username patterns
    ],

    # Output file paths (relative or absolute)
    "output": {
        "structured_resume": "structured_resume.json",
        "pii_vault":         "pii_vault.json",
        "embeddings":        "resume_embeddings.npy",
        "chunks_embeddings": "resume_chunks_embeddings.npy",   # optional
    },

    # Logging
    "log_level": "DEBUG",   # DEBUG | INFO | WARNING | ERROR

    # Optional: generate chunk-level embeddings in addition to full-doc
    "embed_chunks": True,
}

print("✓ CONFIG loaded.")
print(f"  Embedding model : {CONFIG['embedding_model']}")
print(f"  Chunk size      : {CONFIG['chunk_size']} tokens (overlap {CONFIG['chunk_overlap']})")
print(f"  PII fields      : {len(CONFIG['pii_fields'])} categories active")

✓ CONFIG loaded.
  Embedding model : BAAI/bge-base-en-v1.5
  Chunk size      : 300 tokens (overlap 50)
  PII fields      : 10 categories active


---

## Section 2 — Dependency Installation

Run this cell once to install all required packages. Restart the kernel after installation if prompted.

In [3]:
# ────────────────────────────────────────────────────────────────
# DEPENDENCY INSTALLATION
# Run once; restart kernel after if packages were freshly installed.
# ────────────────────────────────────────────────────────────────

import subprocess, sys

_packages = [
    "pymupdf",                   # PDF extraction (fitz)
    "pdfplumber",                # Alternative PDF extraction
    "python-docx",               # DOCX extraction
    "sentence-transformers",     # HuggingFace embeddings
    "numpy",                     # Array operations
    "spacy",                     # NLP helper (NER for names/orgs)
    "unicodedata2",              # Unicode normalisation
    "tqdm",                      # Progress bars
]

print("Installing / verifying dependencies …")
for pkg in _packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗"
    print(f"  {status} {pkg}")

# Download small spaCy model for NER (name detection)
print("\nDownloading spaCy model en_core_web_sm …")
_sp = subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"],
    capture_output=True, text=True
)
if _sp.returncode == 0:
    print("  ✓ en_core_web_sm ready")
else:
    print("  ⚠ spaCy model download failed – name detection may be limited.")
    print("    Run manually: python -m spacy download en_core_web_sm")

print("\nAll dependencies checked. Restart kernel if any were newly installed.")

Installing / verifying dependencies …
  ✓ pymupdf
  ✓ pdfplumber
  ✓ python-docx
  ✓ sentence-transformers
  ✓ numpy
  ✓ spacy
  ✓ unicodedata2
  ✓ tqdm

  ✓ en_core_web_sm ready

All dependencies checked. Restart kernel if any were newly installed.


---

## Section 3 — Imports & Logging Setup

All imports are collected here. A custom PII-safe logging formatter is applied so that even accidental log messages containing common PII patterns are automatically masked before output.

In [4]:
# ────────────────────────────────────────────────────────────────
# IMPORTS & PII-SAFE LOGGING SETUP
# ────────────────────────────────────────────────────────────────

from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import sys
import unicodedata
from pathlib import Path
from typing import Any

import getpass

import numpy as np

# ── Document extraction ──────────────────────────────────────────
try:
    import fitz  # PyMuPDF
    _PYMUPDF_AVAILABLE = True
except ImportError:
    _PYMUPDF_AVAILABLE = False
    logging.warning("PyMuPDF not available — PDF extraction will fall back to pdfplumber.")

try:
    import pdfplumber
    _PDFPLUMBER_AVAILABLE = True
except ImportError:
    _PDFPLUMBER_AVAILABLE = False

try:
    from docx import Document as DocxDocument
    _DOCX_AVAILABLE = True
except ImportError:
    _DOCX_AVAILABLE = False

# ── NLP / Embeddings ─────────────────────────────────────────────
try:
    import spacy
    _nlp = spacy.load("en_core_web_sm")
    _SPACY_AVAILABLE = True
except Exception:
    _SPACY_AVAILABLE = False
    _nlp = None

try:
    from sentence_transformers import SentenceTransformer
    _ST_AVAILABLE = True
except ImportError:
    _ST_AVAILABLE = False


# ── PII-Safe Log Formatter ───────────────────────────────────────

_PII_LOG_PATTERNS = [
    re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b'),   # email
    re.compile(r'\b\+?[\d\s\-().]{7,20}\b'),                                  # phone-like
    re.compile(r'https?://\S+', re.IGNORECASE),                               # URLs
    re.compile(r'www\.\S+', re.IGNORECASE),                                   # www URLs
]

class PIISafeFormatter(logging.Formatter):
    """Scrubs common PII patterns from log messages before emission."""

    def format(self, record: logging.LogRecord) -> str:
        msg = super().format(record)
        for pat in _PII_LOG_PATTERNS:
            msg = pat.sub("[REDACTED]", msg)
        return msg


def _setup_logger(name: str = "resume_pipeline", level: str = "DEBUG") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger   # already configured
    logger.setLevel(getattr(logging, level.upper(), logging.DEBUG))
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(
        PIISafeFormatter(
            fmt="%(asctime)s [%(levelname)-8s] %(name)s — %(message)s",
            datefmt="%H:%M:%S",
        )
    )
    logger.addHandler(handler)
    logger.propagate = False
    return logger


log = _setup_logger(level=CONFIG.get("log_level", "DEBUG"))

log.info("Imports complete.")
log.info(f"PyMuPDF={_PYMUPDF_AVAILABLE}  pdfplumber={_PDFPLUMBER_AVAILABLE}  "
         f"python-docx={_DOCX_AVAILABLE}  spaCy={_SPACY_AVAILABLE}  "
         f"sentence-transformers={_ST_AVAILABLE}")

c:\Users\Dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


21:36:38 [INFO    ] resume_pipeline — Imports complete.
21:36:38 [INFO    ] resume_pipeline — PyMuPDF=True  pdfplumber=True  python-docx=True  spaCy=True  sentence-transformers=True


---

## Section 4 — Salt Provisioning

The hashing salt is resolved in this priority order:

| Priority | Source | Notes |
|---|---|---|
| 1 | `PII_SALT` environment variable | Use in CI / secrets managers |
| 2 | `.pii_salt` file (persisted) | Reused from a previous run |
| 3 | Auto-generated random salt | 256-bit `secrets.token_hex(32)`, saved to `.pii_salt` |

On the **first run** a cryptographically random salt is generated and written to `.pii_salt` (permissions set to `0o600`). Every subsequent run loads the same salt from that file, ensuring all tokens remain consistent across sessions.

> ⚠️ **Add `.pii_salt` to `.gitignore`** — never commit it to version control.  
> ⚠️ **Back it up securely** — losing the salt makes vault tokens irreversible.

In [5]:
# ────────────────────────────────────────────────────────────────
# SALT PROVISIONING
# Priority:
#   1. Environment variable PII_SALT  (CI/secrets manager)
#   2. Existing salt file (.pii_salt) (persisted from a previous run)
#   3. Auto-generate a new cryptographically random salt → save to file
# ────────────────────────────────────────────────────────────────

import secrets

_SALT_FILE = Path(".pii_salt")   # stored next to the notebook; add to .gitignore

def _get_salt() -> str:
    """
    Retrieve or generate the hashing salt.

    Returns the salt as a plain string. Never logs the value itself.
    """
    # ── 1. Environment variable ───────────────────────────────────
    salt = os.environ.get("PII_SALT", "").strip()
    if salt:
        log.info("Salt loaded from environment variable PII_SALT. length=%d", len(salt))
        return salt

    # ── 2. Persisted salt file ────────────────────────────────────
    if _SALT_FILE.exists():
        salt = _SALT_FILE.read_text(encoding="utf-8").strip()
        if salt:
            log.info("Salt loaded from persisted file: %s  length=%d", _SALT_FILE, len(salt))
            return salt
        log.warning("Salt file %s exists but is empty — generating new salt.", _SALT_FILE)

    # ── 3. Generate a new random salt and persist it ──────────────
    salt = secrets.token_hex(32)   # 256-bit random salt → 64-char hex string
    _SALT_FILE.write_text(salt, encoding="utf-8")
    try:
        import stat
        _SALT_FILE.chmod(stat.S_IRUSR | stat.S_IWUSR)   # 0o600 — owner read/write only
    except (AttributeError, NotImplementedError):
        pass   # Windows: chmod not fully supported; use icacls manually if needed

    log.info(
        "New random salt generated and saved → %s  length=%d  [value hidden]",
        _SALT_FILE, len(salt),
    )
    print(f"  ✓ New random salt generated → {_SALT_FILE}")
    print(f"    ⚠  Keep this file secret — add '{_SALT_FILE}' to .gitignore")
    print(f"    ⚠  Losing it means tokens cannot be re-mapped to originals.")
    return salt


# Obtain the salt once; passed explicitly to all hashing functions.
SALT: str = _get_salt()
log.info("Salt provisioned. Length=%d chars. [value hidden]", len(SALT))

21:36:38 [INFO    ] resume_pipeline — New random salt generated and saved → .pii_salt  length=64  [value hidden]
  ✓ New random salt generated → .pii_salt
    ⚠  Keep this file secret — add '.pii_salt' to .gitignore
    ⚠  Losing it means tokens cannot be re-mapped to originals.
21:36:38 [INFO    ] resume_pipeline — Salt provisioned. Length=64 chars. [value hidden]


---

## Section 5 — Text Extraction

### `extract_text(file_path) → str`

Supports:
- **PDF** — tries PyMuPDF first (preserves layout better); falls back to pdfplumber.
- **DOCX** — uses python-docx; extracts paragraphs and table cells in reading order.

Post-extraction, the text is cleaned:
- Unicode normalisation (NFKC)
- Collapsed excess whitespace
- Thinned repeated blank lines
- Preserved structural markers (ALL-CAPS lines treated as section headers)

In [6]:
# ────────────────────────────────────────────────────────────────
# TEXT EXTRACTION
# ────────────────────────────────────────────────────────────────

def _clean_text(raw: str) -> str:
    """
    Normalise and clean extracted text.
    - NFKC unicode normalisation (collapses ligatures, full-width chars, etc.)
    - Collapse horizontal whitespace within lines
    - Thin multiple consecutive blank lines to at most two
    - Strip trailing spaces from each line
    """
    # Unicode normalisation
    text = unicodedata.normalize("NFKC", raw)
    # Standardise line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Strip trailing whitespace per line, collapse intra-line spaces
    lines = [re.sub(r"[ \t]+", " ", line).rstrip() for line in text.split("\n")]
    # Reduce runs of more than 2 consecutive blank lines to exactly 2
    result, blank_count = [], 0
    for line in lines:
        if line == "":
            blank_count += 1
            if blank_count <= 2:
                result.append(line)
        else:
            blank_count = 0
            result.append(line)
    return "\n".join(result).strip()


def _extract_pdf_pymupdf(path: Path) -> str:
    """Extract text from a PDF using PyMuPDF (fitz)."""
    if not _PYMUPDF_AVAILABLE:
        raise ImportError("PyMuPDF (fitz) is not installed.")
    doc = fitz.open(str(path))
    pages = []
    for page in doc:
        pages.append(page.get_text("text"))
    doc.close()
    return "\n".join(pages)


def _extract_pdf_pdfplumber(path: Path) -> str:
    """Extract text from a PDF using pdfplumber."""
    if not _PDFPLUMBER_AVAILABLE:
        raise ImportError("pdfplumber is not installed.")
    pages = []
    with pdfplumber.open(str(path)) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text(x_tolerance=3, y_tolerance=3) or ""
            pages.append(page_text)
    return "\n".join(pages)


def _extract_docx(path: Path) -> str:
    """
    Extract text from a DOCX file using python-docx.
    Reads paragraphs in order and also extracts text from tables.
    """
    if not _DOCX_AVAILABLE:
        raise ImportError("python-docx is not installed.")
    doc = DocxDocument(str(path))
    pieces: list[str] = []

    for block in doc.element.body:
        tag = block.tag.split("}")[-1]  # strip namespace
        if tag == "p":
            # paragraph
            para_text = block.text_content() if hasattr(block, "text_content") else ""
            # Fallback: iterate runs
            if not para_text:
                from docx.oxml.ns import qn
                para_text = "".join(
                    node.text or ""
                    for node in block.iter()
                    if node.tag == qn("w:t")
                )
            pieces.append(para_text)
        elif tag == "tbl":
            # table — flatten cells row by row
            for row in block:
                row_tag = row.tag.split("}")[-1]
                if row_tag != "tr":
                    continue
                row_texts = []
                for cell in row:
                    cell_tag = cell.tag.split("}")[-1]
                    if cell_tag != "tc":
                        continue
                    cell_text = "".join(
                        node.text or ""
                        for node in cell.iter()
                        if node.tag.endswith("}t")
                    )
                    row_texts.append(cell_text.strip())
                pieces.append("  |  ".join(row_texts))

    return "\n".join(pieces)


def extract_text(file_path: str | Path) -> str:
    """
    Extract and clean text from a resume file.

    Parameters
    ----------
    file_path : str | Path
        Absolute or relative path to a .pdf or .docx file.

    Returns
    -------
    str
        Cleaned plain-text content of the resume.

    Raises
    ------
    FileNotFoundError
        If the file does not exist at the given path.
    ValueError
        If the file extension is not .pdf or .docx.
    RuntimeError
        If extraction fails with all available backends.
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Resume file not found: {path}")

    suffix = path.suffix.lower()
    log.debug("Extracting text from: %s (format=%s)", path.name, suffix)

    raw_text = ""
    if suffix == ".pdf":
        if _PYMUPDF_AVAILABLE:
            try:
                raw_text = _extract_pdf_pymupdf(path)
                log.debug("PDF extracted via PyMuPDF. Raw chars=%d", len(raw_text))
            except Exception as exc:
                log.warning("PyMuPDF extraction failed (%s); falling back to pdfplumber.", exc)
                raw_text = ""

        if not raw_text.strip() and _PDFPLUMBER_AVAILABLE:
            raw_text = _extract_pdf_pdfplumber(path)
            log.debug("PDF extracted via pdfplumber. Raw chars=%d", len(raw_text))

        if not raw_text.strip():
            raise RuntimeError(
                "PDF text extraction failed with all available backends. "
                "Install PyMuPDF: pip install pymupdf"
            )

    elif suffix in (".docx", ".doc"):
        raw_text = _extract_docx(path)
        log.debug("DOCX extracted. Raw chars=%d", len(raw_text))

    else:
        raise ValueError(
            f"Unsupported file format '{suffix}'. Supported formats: .pdf, .docx"
        )

    cleaned = _clean_text(raw_text)
    log.info("Text extraction complete. Cleaned length=%d chars.", len(cleaned))
    return cleaned


print("✓ extract_text() defined.")

✓ extract_text() defined.


---

## Section 6 — Structured Schema Parsing

### `parse_structured_schema(text) → dict`

Converts free-form resume text into a typed JSON schema using rule-based parsing:
- **Section detection** via `ALL CAPS` markers or known keyword headers (e.g., `EDUCATION`, `EXPERIENCE`)
- **Field extraction** using targeted regex patterns for each section
- **Profile link normalisation** — detected profile URLs are stored as `{"type": ..., "token": ..., "raw": ...}` placeholder objects; tokenization happens in the PII stage

Output schema structure:
```json
{
  "personal_info": { "name": "...", "email": "...", "phone": "...", "address": "..." },
  "education": [ { "institution": "...", "degree": "...", "field": "...", "dates": "...", "gpa": "..." } ],
  "skills": { "technical": [], "soft": [], "tools": [], "languages": [] },
  "experience": [ { "title": "...", "company": "...", "dates": "...", "location": "...", "bullets": [] } ],
  "projects": [ { "name": "...", "description": "...", "technologies": [], "links": [] } ],
  "profiles": [ { "type": "github|linkedin|portfolio|other", "raw_url": "..." } ]
}
```

In [7]:
# ────────────────────────────────────────────────────────────────
# STRUCTURED SCHEMA PARSING
# ────────────────────────────────────────────────────────────────

# ── Section header detection ─────────────────────────────────────

_SECTION_KEYWORDS = {
    "personal_info":  ["contact", "personal information", "personal info", "about me"],
    "education":      ["education", "academic background", "qualifications", "academic"],
    "skills":         ["skills", "technical skills", "competencies", "technologies",
                       "expertise", "proficiencies"],
    "experience":     ["experience", "work experience", "employment", "work history",
                       "professional experience", "career"],
    "projects":       ["projects", "personal projects", "notable projects",
                       "key projects", "portfolio"],
    "profiles":       ["profiles", "links", "online profiles", "social", "portfolio links"],
    "certifications": ["certifications", "certificates", "licenses", "accreditations"],
    "summary":        ["summary", "objective", "profile", "professional summary",
                       "career objective", "about"],
    "awards":         ["awards", "honors", "achievements", "accomplishments"],
    "publications":   ["publications", "papers", "research"],
    "languages":      ["languages", "spoken languages"],
    "interests":      ["interests", "hobbies", "activities"],
    "references":     ["references"],
}

def _detect_section(line: str) -> str | None:
    """Return the canonical section name if a line is a section header, else None."""
    stripped = line.strip()
    # ALL-CAPS line (≥3 chars, no digits-only)
    candidate = stripped.lower().rstrip(":").strip()
    for section, keywords in _SECTION_KEYWORDS.items():
        for kw in keywords:
            if candidate == kw or stripped.upper() == stripped and candidate in kw:
                return section
    # fuzzy: line is ≤5 words, all caps or title case, matches a keyword
    if len(stripped.split()) <= 5:
        for section, keywords in _SECTION_KEYWORDS.items():
            if any(kw in candidate for kw in keywords):
                return section
    return None


def _split_into_sections(text: str) -> dict[str, str]:
    """
    Split resume text into raw section blobs keyed by canonical section name.
    Lines that match no known section are added to 'personal_info' (header region).
    """
    sections: dict[str, list[str]] = {"personal_info": []}
    current = "personal_info"

    for line in text.split("\n"):
        detected = _detect_section(line)
        if detected:
            current = detected
            if current not in sections:
                sections[current] = []
        else:
            sections.setdefault(current, []).append(line)

    return {k: "\n".join(v).strip() for k, v in sections.items() if v}


# ── Field-level parsers ───────────────────────────────────────────

_EMAIL_RE    = re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b')
_PHONE_RE    = re.compile(
    r'(?<!\d)'
    r'(\+?1?\s?)?'
    r'(\(?\d{2,4}\)?[\s.\-]?)'
    r'(\d{3,4}[\s.\-]?)'
    r'(\d{4})'
    r'(\s?(ext|x|ext\.)\s?\d{1,5})?'
    r'(?!\d)',
    re.IGNORECASE,
)
_URL_RE      = re.compile(r'https?://[^\s<>"\']+|www\.[^\s<>"\']+', re.IGNORECASE)
_PROFILE_RES = {
    "github":        re.compile(r'(https?://)?(www\.)?github\.com/[\w\-/.]+', re.IGNORECASE),
    "gitlab":        re.compile(r'(https?://)?(www\.)?gitlab\.com/[\w\-/.]+', re.IGNORECASE),
    "linkedin":      re.compile(r'(https?://)?(www\.)?linkedin\.com/in/[\w\-/.]+', re.IGNORECASE),
    "stackoverflow": re.compile(r'(https?://)?(www\.)?stackoverflow\.com/users/[\w\-/.]+', re.IGNORECASE),
    "twitter":       re.compile(r'(https?://)?(www\.)?twitter\.com/[\w\-]+', re.IGNORECASE),
    "medium":        re.compile(r'(https?://)?(www\.)?medium\.com/@?[\w\-/.]+', re.IGNORECASE),
    "kaggle":        re.compile(r'(https?://)?(www\.)?kaggle\.com/[\w\-/.]+', re.IGNORECASE),
}
_HANDLE_RE   = re.compile(r'@[A-Za-z0-9_]{2,30}\b')

def _extract_personal_info(blob: str) -> dict[str, str]:
    info: dict[str, str] = {}
    emails = _EMAIL_RE.findall(blob)
    if emails:
        info["email"] = emails[0]
    phones = _PHONE_RE.findall(blob)
    if phones:
        info["phone"] = "".join(phones[0]).strip()
    # Name heuristic: first non-empty line that is not an email/phone/URL
    for line in blob.split("\n"):
        line = line.strip()
        if (line
                and not _EMAIL_RE.search(line)
                and not _PHONE_RE.search(line)
                and not _URL_RE.search(line)
                and len(line.split()) <= 6
                and any(c.isalpha() for c in line)):
            info["name"] = line
            break
    # Address: line containing numeric prefix or known address words
    addr_re = re.compile(
        r'\b(\d{1,5}\s[\w\s.,-]{3,60}|'
        r'(street|st\.?|avenue|ave\.?|road|rd\.?|blvd|lane|ln\.?|drive|dr\.?|'
        r'city|state|zip|pincode|postal)\b)',
        re.IGNORECASE,
    )
    for line in blob.split("\n"):
        if addr_re.search(line):
            info["address"] = line.strip()
            break
    return info


def _extract_education(blob: str) -> list[dict]:
    """
    Heuristic education extraction.
    Looks for degree keywords and institution patterns.
    """
    degree_re = re.compile(
        r'\b(B\.?Tech|M\.?Tech|B\.?E\.?|M\.?E\.?|B\.?Sc\.?|M\.?Sc\.?|'
        r'B\.?A\.?|M\.?A\.?|Ph\.?D\.?|MBA|BBA|BCA|MCA|'
        r'Bachelor|Master|Doctor|Associate|Diploma|Certificate)\b',
        re.IGNORECASE,
    )
    date_re  = re.compile(r'\b(19|20)\d{2}\b')
    gpa_re   = re.compile(r'\b(GPA|CGPA|Score|Grade)[:\s]*([\d.]+)', re.IGNORECASE)

    entries, current = [], {}
    for line in blob.split("\n"):
        line = line.strip()
        if not line:
            if current:
                entries.append(current)
                current = {}
            continue
        dm = degree_re.search(line)
        if dm:
            if current:
                entries.append(current)
                current = {}
            current["degree"] = dm.group(0)
            current["description"] = line
        dates = date_re.findall(line)
        if dates:
            current.setdefault("dates", " – ".join(sorted(set(dates))))
        gm = gpa_re.search(line)
        if gm:
            current["gpa"] = gm.group(2)
        if line and "institution" not in current and not dm:
            current.setdefault("institution", line)
    if current:
        entries.append(current)
    return entries or [{"raw": blob}]


def _extract_skills(blob: str) -> dict[str, list[str]]:
    """
    Parse skills section into sub-categories.
    Falls back to returning a flat list under 'general'.
    """
    cat_re = re.compile(
        r'^(technical|programming|languages?|tools?|frameworks?|'
        r'soft\s*skills?|interpersonal|databases?|cloud|devops|'
        r'platforms?|other)[:\s]+',
        re.IGNORECASE,
    )
    skills: dict[str, list[str]] = {"technical": [], "tools": [], "soft": [], "languages": [], "other": []}
    current_cat = "other"

    for line in blob.split("\n"):
        line = line.strip()
        if not line:
            continue
        m = cat_re.match(line)
        if m:
            cat_raw = m.group(1).lower().strip()
            if "language" in cat_raw:
                current_cat = "languages"
            elif "tool" in cat_raw or "framework" in cat_raw or "platform" in cat_raw:
                current_cat = "tools"
            elif "soft" in cat_raw or "interpersonal" in cat_raw:
                current_cat = "soft"
            else:
                current_cat = "technical"
            remainder = line[m.end():]
            items = [s.strip() for s in re.split(r'[,;|•·\u2022]', remainder) if s.strip()]
            skills[current_cat].extend(items)
        else:
            items = [s.strip() for s in re.split(r'[,;|•·\u2022]', line) if s.strip()]
            skills[current_cat].extend(items)

    # Deduplicate
    return {k: list(dict.fromkeys(v)) for k, v in skills.items() if v}


def _extract_experience(blob: str) -> list[dict]:
    """
    Heuristic work-experience extraction.
    Identifies job entries by date range patterns and bullet collections.
    """
    date_range_re = re.compile(
        r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|January|February|'
        r'March|April|June|July|August|September|October|November|December)?'
        r'\s*(20|19)?\d{2}\s*[-–—]\s*'
        r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|January|February|'
        r'March|April|June|July|August|September|October|November|December|'
        r'Present|Current|Till date|Now)?\s*(20|19)?\d{0,4}',
        re.IGNORECASE,
    )
    bullet_re = re.compile(r'^[-•*▸►·\u2022]\s*(.+)')

    entries, current = [], {}
    for line in blob.split("\n"):
        stripped = line.strip()
        if not stripped:
            continue
        drm = date_range_re.search(stripped)
        bm  = bullet_re.match(stripped)
        if drm and not bm:
            if current:
                entries.append(current)
            current = {"dates": drm.group(0).strip(), "bullets": [], "raw_header": stripped}
        elif bm:
            current.setdefault("bullets", []).append(bm.group(1).strip())
        else:
            if current and "title" not in current:
                current["title"] = stripped
            elif current:
                current.setdefault("company", stripped)
    if current:
        entries.append(current)
    return entries or [{"raw": blob}]


def _extract_projects(blob: str) -> list[dict]:
    tech_re  = re.compile(
        r'(Technologies|Tech Stack|Built with|Tools|Stack)[:\s]+(.+)',
        re.IGNORECASE,
    )
    url_re   = _URL_RE
    bullet_re = re.compile(r'^[-•*▸►·\u2022]\s*(.+)')

    entries, current = [], {}
    for line in blob.split("\n"):
        stripped = line.strip()
        if not stripped:
            if current:
                entries.append(current)
                current = {}
            continue
        tm = tech_re.search(stripped)
        bm = bullet_re.match(stripped)
        um = url_re.search(stripped)
        if tm:
            techs = [t.strip() for t in re.split(r'[,;|]', tm.group(2)) if t.strip()]
            current.setdefault("technologies", []).extend(techs)
        elif um:
            current.setdefault("links", []).append(um.group(0))
        elif bm:
            current.setdefault("bullets", []).append(bm.group(1).strip())
        else:
            if "name" not in current:
                current["name"] = stripped
            else:
                current.setdefault("description", "")
                current["description"] = (current["description"] + " " + stripped).strip()
    if current:
        entries.append(current)
    return entries or [{"raw": blob}]


def _extract_profiles(blob: str, all_text: str = "") -> list[dict[str, str]]:
    """Extract profile URLs from the profiles blob and optionally the entire text."""
    scan_text = blob + "\n" + all_text
    found: list[dict[str, str]] = []
    seen_urls: set[str] = set()

    for ptype, pat in _PROFILE_RES.items():
        for m in pat.finditer(scan_text):
            url = m.group(0).strip()
            if url not in seen_urls:
                seen_urls.add(url)
                found.append({"type": ptype, "raw_url": url})

    # Generic URLs not already captured
    for m in _URL_RE.finditer(scan_text):
        url = m.group(0).strip()
        if url not in seen_urls:
            seen_urls.add(url)
            found.append({"type": "website", "raw_url": url})

    return found


# ── Main schema parser ────────────────────────────────────────────

def parse_structured_schema(text: str) -> dict:
    """
    Parse free-form resume text into a structured JSON-compatible dictionary.

    Parameters
    ----------
    text : str
        Cleaned resume text (output of extract_text()).

    Returns
    -------
    dict
        Structured schema with keys:
        personal_info, education, skills, experience, projects, profiles, summary, raw_text.
    """
    log.info("Parsing structured schema …")
    sections = _split_into_sections(text)
    log.debug("Detected sections: %s", list(sections.keys()))

    schema: dict[str, Any] = {
        "personal_info": {},
        "summary":       "",
        "education":     [],
        "skills":        {},
        "experience":    [],
        "projects":      [],
        "profiles":      [],
        "raw_sections":  {},
        "_metadata": {
            "parser_version": "1.0",
            "sections_detected": list(sections.keys()),
        },
    }

    # personal_info
    pi_blob = sections.get("personal_info", "")
    header_blob = "\n".join(text.split("\n")[:20])  # first 20 lines often contain contact info
    schema["personal_info"] = _extract_personal_info(pi_blob or header_blob)

    # summary
    schema["summary"] = sections.get("summary", "")

    # education
    if "education" in sections:
        schema["education"] = _extract_education(sections["education"])

    # skills
    if "skills" in sections:
        schema["skills"] = _extract_skills(sections["skills"])

    # experience
    if "experience" in sections:
        schema["experience"] = _extract_experience(sections["experience"])

    # projects
    if "projects" in sections:
        schema["projects"] = _extract_projects(sections["projects"])

    # profiles — scan dedicated section + entire text
    schema["profiles"] = _extract_profiles(
        sections.get("profiles", ""),
        all_text=text,
    )

    # raw sections (for debugging / Fall 2 ingestion)
    schema["raw_sections"] = {k: v for k, v in sections.items()}

    log.info(
        "Schema parsed. personal_info keys=%s | experience=%d | "
        "education=%d | projects=%d | profiles=%d",
        list(schema["personal_info"].keys()),
        len(schema["experience"]),
        len(schema["education"]),
        len(schema["projects"]),
        len(schema["profiles"]),
    )
    return schema


print("✓ parse_structured_schema() defined.")

✓ parse_structured_schema() defined.


---

## Section 7 — PII Detection

### `detect_pii(text, config, salt) → list[dict]`

Covers all detection categories defined in `CONFIG["pii_fields"]`:

| Category | Patterns Detected |
|---|---|
| `name` | Full names via spaCy NER (`PERSON`) + heuristics |
| `email` | RFC-5322 variants + obfuscated forms (`[at]`, `(dot)`) |
| `phone` | All country formats, separators, extensions |
| `address` | Street addresses, city/postal patterns |
| `government_id` | SSN, Aadhaar, PAN, Passport, NINO |
| `demographics` | DOB dates, gender/nationality/religion keywords |
| `financial` | Bank account/IFSC, salary/CTC figures, tax IDs |
| `urls` | http(s)://, www., URL shorteners |
| `profiles` | GitHub, GitLab, LinkedIn, StackOverflow, etc. |
| `social_handles` | @username patterns |

Each detection is returned as a dict:
```python
{
  "original": "...",   # raw matched text
  "canonical": "...",  # normalized for consistent hashing
  "type": "email",     # category
  "start": 42,         # char offset in source
  "end": 63
}
```

In [8]:
# ────────────────────────────────────────────────────────────────
# PII DETECTION
# ────────────────────────────────────────────────────────────────

# ── Pattern registry ──────────────────────────────────────────────

_PII_PATTERNS: dict[str, list[re.Pattern]] = {

    "email": [
        # Standard email
        re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b'),
        # Obfuscated: john [at] example [dot] com or john(at)example(dot)com
        re.compile(
            r'\b[A-Za-z0-9._%+\-]+'
            r'\s*[\[(]?\s*(?:at|@)\s*[\])]?\s*'
            r'[A-Za-z0-9.\-]+'
            r'\s*[\[(]?\s*(?:dot|\.)\s*[\])]?\s*'
            r'[A-Za-z]{2,}\b',
            re.IGNORECASE,
        ),
        # john(dot)doe(at)gmail(dot)com
        re.compile(
            r'\b[A-Za-z0-9]+(?:\(dot\)[A-Za-z0-9]+)*'
            r'\(at\)[A-Za-z0-9]+(?:\(dot\)[A-Za-z0-9]+)+\b',
            re.IGNORECASE,
        ),
    ],

    "phone": [
        # International: +1 (555) 555-5555, 555.555.5555, +91-9876543210, etc.
        re.compile(
            r'(?<!\d)'
            r'(\+?(\d{1,3})[\s\-.]?)?'    # country code
            r'(\(?\d{1,4}\)?[\s.\-]?)'    # area code
            r'(\d{2,4}[\s.\-]?){2,3}'     # subscriber
            r'(\s?(ext|x|ext\.)\s?\d{1,5})?'
            r'(?!\d)',
            re.IGNORECASE,
        ),
    ],

    "address": [
        # Heuristic: starts with a number followed by road/street terms
        re.compile(
            r'\b\d{1,5}\s+[\w\s.,-]{3,60}'
            r'(?:street|st\.?|avenue|ave\.?|road|rd\.?|blvd\.?|'
            r'lane|ln\.?|drive|dr\.?|court|ct\.?|place|pl\.?|'
            r'way|circle|terrace|parkway|highway|hwy)\b',
            re.IGNORECASE,
        ),
        # Postal code patterns: US ZIP, UK, CA, IN, AU
        re.compile(
            r'\b('
            r'\d{5}(-\d{4})?'            # US ZIP
            r'|[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}'  # UK postcode
            r'|\d{6}'                     # India PIN
            r'|[A-Z]\d[A-Z]\s?\d[A-Z]\d' # Canada
            r'|\d{4}'                     # Australia
            r')\b',
        ),
    ],

    "government_id": [
        # US SSN: 123-45-6789 or 123456789
        re.compile(r'\b\d{3}[-\s]?\d{2}[-\s]?\d{4}\b'),
        # India Aadhaar: XXXX XXXX XXXX
        re.compile(r'\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b'),
        # India PAN: ABCDE1234F
        re.compile(r'\b[A-Z]{5}\d{4}[A-Z]\b'),
        # UK NINO: AA 99 99 99 A
        re.compile(r'\b[A-Z]{2}\s?\d{2}\s?\d{2}\s?\d{2}\s?[A-D]\b', re.IGNORECASE),
        # Passport: generic alphanumeric (context-aware min 7 chars)
        re.compile(r'\b(?:passport\s*(?:no|number|#)?[:.\s]*)?[A-Z]{1,2}\d{7,8}\b', re.IGNORECASE),
    ],

    "demographics": [
        # Date of Birth patterns
        re.compile(
            r'\b(DOB|Date of Birth|Born|Birth\s*Date)[:\s]*'
            r'\d{1,2}[\s/\-]\d{1,2}[\s/\-]\d{2,4}\b',
            re.IGNORECASE,
        ),
        # Full date patterns that might be DOB
        re.compile(
            r'\b\d{1,2}\s*(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s*\d{4}\b',
            re.IGNORECASE,
        ),
        # Gender
        re.compile(
            r'\b(Gender|Sex)\s*[:\-]?\s*(Male|Female|Non[\s\-]?binary|'
            r'Transgender|Prefer not to say)\b',
            re.IGNORECASE,
        ),
        # Nationality / Religion / Marital status
        re.compile(
            r'\b(Nationality|Citizenship|Religion|Marital\s*Status|'
            r'Caste|Community)\s*[:\-]?\s*[A-Za-z\s]{2,30}\b',
            re.IGNORECASE,
        ),
    ],

    "financial": [
        # Salary / CTC
        re.compile(
            r'\b(Salary|CTC|Package|Compensation|Gross|Net|'
            r'Expected\s*Salary|Current\s*CTC)[:\s]+'
            r'[\$£€₹\d.,LKlk\s]+(?:per\s*(?:annum|month|year|pa))?',
            re.IGNORECASE,
        ),
        # Bank account numbers (generic 8–18 digits)
        re.compile(r'\b\d{8,18}\b'),
        # IFSC code (India)
        re.compile(r'\b[A-Z]{4}0[A-Z0-9]{6}\b'),
        # PAN (also in gov IDs but captured here for financial context)
        re.compile(r'\b[A-Z]{5}\d{4}[A-Z]\b'),
    ],

    "urls": [
        re.compile(r'https?://[^\s<>"\'\]]+', re.IGNORECASE),
        re.compile(r'www\.[a-zA-Z0-9\-]+\.[a-zA-Z]{2,}[^\s<>"\'\]]*', re.IGNORECASE),
        # URL shorteners
        re.compile(r'\b(?:bit\.ly|t\.co|tinyurl\.com|goo\.gl|ow\.ly)/[A-Za-z0-9\-_]+\b', re.IGNORECASE),
    ],

    "profiles": [_pat for _pat in _PROFILE_RES.values()],

    "social_handles": [
        re.compile(r'(?<!\w)@[A-Za-z0-9_]{2,30}\b'),
    ],

    "name": [],  # handled via spaCy NER + heuristics below
}


# ── Helper: canonicalize a detected value ────────────────────────

def _canonicalize(value: str, pii_type: str) -> str:
    """Normalize a detected PII value for consistent hashing."""
    v = value.strip()
    if pii_type in ("email", "profiles", "urls", "social_handles"):
        v = v.lower()
    # Remove UTM params and anchors from URLs
    if pii_type in ("urls", "profiles"):
        v = re.sub(r'[?#].+$', '', v)   # strip query string / fragment
        v = v.rstrip("/")
        v = re.sub(r'^https?://(www\.)?', '', v)   # strip scheme + www
        v = v.lower()
    if pii_type in ("phone",):
        v = re.sub(r'[\s\-.()+]', '', v)
    if pii_type == "government_id":
        v = re.sub(r'[\s\-]', '', v).upper()
    return v


# ── Name detection via spaCy ─────────────────────────────────────

def _detect_names_spacy(text: str) -> list[dict]:
    """Use spaCy NER to find PERSON entities."""
    if not _SPACY_AVAILABLE or _nlp is None:
        log.debug("spaCy unavailable — skipping NER name detection.")
        return []
    doc = _nlp(text[:50000])  # cap to 50k chars for performance
    results = []
    for ent in doc.ents:
        if ent.label_ == "PERSON" and len(ent.text.split()) >= 2:
            results.append({
                "original":  ent.text,
                "canonical": ent.text.strip().lower(),
                "type":      "name",
                "start":     ent.start_char,
                "end":       ent.end_char,
            })
    return results


# ── Profile type classifier ───────────────────────────────────────

def _classify_profile_type(url: str) -> str:
    """Return a fine-grained profile type label for a URL."""
    url_lower = url.lower()
    if "github.com" in url_lower:
        return "github"
    if "gitlab.com" in url_lower:
        return "gitlab"
    if "linkedin.com" in url_lower:
        return "linkedin"
    if "stackoverflow.com" in url_lower:
        return "stackoverflow"
    if "twitter.com" in url_lower or "x.com" in url_lower:
        return "twitter"
    if "medium.com" in url_lower:
        return "medium"
    if "kaggle.com" in url_lower:
        return "kaggle"
    if "bitbucket.org" in url_lower:
        return "bitbucket"
    return "website"


# ── Main PII detector ─────────────────────────────────────────────

def detect_pii(
    text: str,
    config: dict | None = None,
    salt: str = "",
) -> list[dict]:
    """
    Detect all PII instances in the given text.

    Parameters
    ----------
    text   : str   — The text to scan.
    config : dict  — Pipeline config (uses CONFIG global if None).
    salt   : str   — Hashing salt (required for determinism; not used in detection itself).

    Returns
    -------
    list[dict]
        Each entry: {original, canonical, type, start, end}
        Sorted by start offset, deduplicated by span overlap.
    """
    cfg = config or CONFIG
    active_fields: set[str] = set(cfg.get("pii_fields", list(_PII_PATTERNS.keys())))
    detections: list[dict] = []

    # ── regex-based detection ────────────────────────────────────
    for field, patterns in _PII_PATTERNS.items():
        if field not in active_fields or field == "name":
            continue
        for pat in patterns:
            for m in pat.finditer(text):
                matched = m.group(0).strip()
                if not matched or len(matched) < 2:
                    continue
                # Refine type for profile URLs
                det_type = field
                if field == "profiles":
                    det_type = _classify_profile_type(matched)
                elif field == "urls":
                    # Re-classify generic URLs that are actually profiles
                    reclassified = _classify_profile_type(matched)
                    if reclassified != "website":
                        det_type = reclassified

                detections.append({
                    "original":  matched,
                    "canonical": _canonicalize(matched, det_type),
                    "type":      det_type,
                    "start":     m.start(),
                    "end":       m.end(),
                })

    # ── spaCy NER for names ───────────────────────────────────────
    if "name" in active_fields:
        name_hits = _detect_names_spacy(text)
        detections.extend(name_hits)

    # ── Sort by start offset ──────────────────────────────────────
    detections.sort(key=lambda d: d["start"])

    # ── Deduplicate overlapping spans ─────────────────────────────
    # Keep the longest span when two detections overlap; prefer more specific types.
    _TYPE_PRIORITY = {
        "github": 1, "gitlab": 1, "linkedin": 1, "stackoverflow": 1,
        "twitter": 1, "medium": 1, "kaggle": 1, "bitbucket": 1,
        "email": 2, "phone": 2, "name": 2, "government_id": 2,
        "demographics": 3, "financial": 3, "social_handles": 2,
        "urls": 4, "website": 4, "address": 5,
    }
    deduped: list[dict] = []
    for d in detections:
        if not deduped:
            deduped.append(d)
            continue
        prev = deduped[-1]
        # Check overlap
        if d["start"] < prev["end"]:
            # Keep the one with higher priority (lower number) or longer span
            p_pri = _TYPE_PRIORITY.get(prev["type"], 99)
            d_pri = _TYPE_PRIORITY.get(d["type"], 99)
            if d_pri < p_pri:
                deduped[-1] = d   # replace with higher-priority detection
            elif d_pri == p_pri and (d["end"] - d["start"]) > (prev["end"] - prev["start"]):
                deduped[-1] = d   # replace with longer match
            # else keep prev
        else:
            deduped.append(d)

    log.info(
        "PII detection complete. %d raw detections → %d after dedup.",
        len(detections), len(deduped),
    )
    # Mask originals in log output
    for d in deduped:
        masked = d["original"][:2] + "***" + d["original"][-2:]
        log.debug("  [%s] %s", d["type"], masked)

    return deduped


print("✓ detect_pii() defined.")

✓ detect_pii() defined.


---

## Section 8 — PII Tokenization & Vault

### `tokenize_pii(detections, salt) → (sanitized_text, vault_entries)`

Replaces each detected PII span with a typed token:

```
<EMAIL_1_a3f92b...>   <NAME_1_7c4d12...>   <GITHUB_1_88ef01...>
```

Token generation rules:
- **Token index** (`_1_`, `_2_`, …) is per-type and incremented for each new unique original value.
- **Hash** = `SHA-256(salt + canonical_value)` — first 16 hex chars used in the token.
- **Same original always → same token** (deterministic, salt-bound).
- Replacements are applied right-to-left to preserve character offsets.

### `save_vault(vault_entries, out_path)`

Writes `pii_vault.json` containing the full token↔original mapping. Format:
```json
[
  {
    "token": "<GITHUB_1_a3f92b01c2d3e4f5>",
    "original": "https://github.com/username",
    "canonical": "github.com/username",
    "hash": "a3f92b01c2d3e4f5...",
    "type": "github"
  }
]
```

### `restore_pii(sanitized_text, vault_path) → str`

**Privileged operation.** Replaces all tokens in the text with their original values using the vault. Requires the vault file to be accessible and the caller to have explicit authorization.

In [9]:
# ────────────────────────────────────────────────────────────────
# PII TOKENIZATION & VAULT MANAGEMENT
# ────────────────────────────────────────────────────────────────

# ── Type → token prefix map ───────────────────────────────────────

_TOKEN_PREFIX: dict[str, str] = {
    "name":           "NAME",
    "email":          "EMAIL",
    "phone":          "PHONE",
    "address":        "ADDR",
    "government_id":  "GOVID",
    "demographics":   "DEMO",
    "financial":      "FIN",
    "urls":           "URL",
    "website":        "URL",
    "profiles":       "URL",
    "github":         "GITHUB",
    "gitlab":         "GITLAB",
    "linkedin":       "LINKEDIN",
    "stackoverflow":  "SOFLOW",
    "twitter":        "TWITTER",
    "medium":         "MEDIUM",
    "kaggle":         "KAGGLE",
    "bitbucket":      "BITBUCKET",
    "social_handles": "HANDLE",
}


def _make_hash(salt: str, canonical: str) -> str:
    """
    Compute deterministic SHA-256 hash.
    Formula: SHA256(salt + canonicalized_original_value)
    Returns full hex digest (64 chars).
    """
    combined = (salt + canonical).encode("utf-8")
    return hashlib.sha256(combined).hexdigest()


def _make_token(pii_type: str, index: int, hash_value: str) -> str:
    """
    Construct a typed token.
    Format: <PREFIX_INDEX_HASH16>
    e.g. <GITHUB_1_a3f92b01c2d3e4f5>
    """
    prefix = _TOKEN_PREFIX.get(pii_type, "PII")
    short_hash = hash_value[:16]
    return f"<{prefix}_{index}_{short_hash}>"


def tokenize_pii(
    text: str,
    detections: list[dict],
    salt: str,
) -> tuple[str, list[dict]]:
    """
    Replace PII spans in text with deterministic typed tokens.

    Parameters
    ----------
    text       : str         — Original text containing PII.
    detections : list[dict]  — Output of detect_pii().
    salt       : str         — Hashing salt.

    Returns
    -------
    sanitized_text : str
        Text with all detected PII replaced by tokens.
    vault_entries : list[dict]
        List of vault records (token ↔ original mappings).
    """
    if not detections:
        log.info("No PII detections; text unchanged.")
        return text, []

    # ── Build canonical → token mapping (for determinism) ────────
    # Multiple occurrences of the same canonical value → same token
    canonical_to_token: dict[str, str]  = {}
    canonical_to_hash:  dict[str, str]  = {}
    type_counters:      dict[str, int]  = {}   # per-type sequential index
    vault_entries:      list[dict]      = []

    for det in detections:
        ckey = (det["type"], det["canonical"])
        if ckey not in canonical_to_token:
            h = _make_hash(salt, det["canonical"])
            canonical_to_hash[ckey] = h
            type_counters[det["type"]] = type_counters.get(det["type"], 0) + 1
            idx = type_counters[det["type"]]
            token = _make_token(det["type"], idx, h)
            canonical_to_token[ckey] = token
            vault_entries.append({
                "token":     token,
                "original":  det["original"],
                "canonical": det["canonical"],
                "hash":      h,
                "type":      det["type"],
            })
            log.debug(
                "  New token created: type=%s index=%d token=%s",
                det["type"], idx, token,
            )
        else:
            log.debug(
                "  Reusing existing token for duplicate %s value.",
                det["type"],
            )

    # ── Apply replacements right-to-left (preserves offsets) ─────
    # Sort detections by start descending
    ordered = sorted(detections, key=lambda d: d["start"], reverse=True)
    text_chars = list(text)

    for det in ordered:
        token = canonical_to_token[(det["type"], det["canonical"])]
        text_chars[det["start"]:det["end"]] = list(token)

    sanitized_text = "".join(text_chars)

    log.info(
        "Tokenization complete. %d unique PII values replaced. "
        "Vault entries=%d.",
        len(vault_entries), len(vault_entries),
    )
    return sanitized_text, vault_entries


# ── Vault I/O ────────────────────────────────────────────────────

def save_vault(vault_entries: list[dict], out_path: str | Path) -> None:
    """
    Persist the PII vault to disk as JSON.

    Parameters
    ----------
    vault_entries : list[dict]  — Output of tokenize_pii().
    out_path      : str | Path  — Destination file path.

    Security note
    -------------
    Immediately after saving, restrict file permissions:
      Linux/macOS : chmod 600 pii_vault.json
      Windows     : icacls pii_vault.json /inheritance:r /grant:r "%USERNAME%:F"
    """
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    existing: list[dict] = []
    if out.exists():
        try:
            with open(out, "r", encoding="utf-8") as f:
                existing = json.load(f)
            log.debug("Loaded %d existing vault entries from %s.", len(existing), out.name)
        except (json.JSONDecodeError, OSError) as exc:
            log.warning("Could not read existing vault (%s); overwriting.", exc)
            existing = []

    # Merge: avoid duplicating tokens
    existing_tokens = {e["token"] for e in existing}
    new_entries = [e for e in vault_entries if e["token"] not in existing_tokens]
    merged = existing + new_entries

    with open(out, "w", encoding="utf-8") as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)

    log.info(
        "Vault saved → %s  (%d new + %d existing = %d total entries).",
        out, len(new_entries), len(existing), len(merged),
    )
    print(f"  ⚠  Vault saved to: {out}")
    print("  ⚠  IMPORTANT: Restrict file permissions immediately:")
    print("       Linux/macOS : chmod 600", out.name)
    print("       Windows     : icacls", out.name, '/inheritance:r /grant:r "%USERNAME%:F"')


def load_vault(vault_path: str | Path) -> dict[str, str]:
    """
    Load vault and return a token → original mapping dict.

    Parameters
    ----------
    vault_path : str | Path  — Path to pii_vault.json.

    Returns
    -------
    dict[str, str]
        Maps token strings to their original PII values.
    """
    path = Path(vault_path)
    if not path.exists():
        raise FileNotFoundError(f"Vault file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        entries = json.load(f)
    return {e["token"]: e["original"] for e in entries}


def restore_pii(
    sanitized_text: str,
    vault_path: str | Path,
    authorized: bool = False,
) -> str:
    """
    PRIVILEGED OPERATION — Restore original PII values from a sanitized text.

    Parameters
    ----------
    sanitized_text : str         — Text containing PII tokens.
    vault_path     : str | Path  — Path to pii_vault.json.
    authorized     : bool        — Must be explicitly True; prevents accidental calls.

    Returns
    -------
    str  — Text with tokens replaced by original values.

    Raises
    ------
    PermissionError  — If authorized=False.
    FileNotFoundError — If vault not found.
    """
    if not authorized:
        raise PermissionError(
            "restore_pii() requires authorized=True. "
            "This is a privileged operation — ensure caller is authorized before proceeding."
        )
    mapping = load_vault(vault_path)
    restored = sanitized_text
    for token, original in mapping.items():
        restored = restored.replace(token, original)
    log.info("PII restored for %d tokens. [authorized call]", len(mapping))
    return restored


# ── Schema tokenizer: walk schema dict and tokenize string values ──

def _tokenize_value(value: str, tokenize_fn) -> str:
    """Apply the tokenize function to a single string value."""
    sanitized, _ = tokenize_fn(value, detect_pii(value, CONFIG, SALT), SALT)
    return sanitized


def sanitize_schema(schema: dict, salt: str) -> tuple[dict, list[dict]]:
    """
    Walk the structured schema dict recursively and tokenize all PII string values.

    Returns
    -------
    sanitized_schema : dict
        Schema with PII values replaced by tokens.
    all_vault_entries : list[dict]
        Combined vault entries from all fields.
    """
    all_vault: list[dict] = []

    def _walk(obj: Any) -> Any:
        if isinstance(obj, str):
            dets = detect_pii(obj, CONFIG, salt)
            if dets:
                sanitized, v_entries = tokenize_pii(obj, dets, salt)
                all_vault.extend(v_entries)
                return sanitized
            return obj
        if isinstance(obj, list):
            return [_walk(item) for item in obj]
        if isinstance(obj, dict):
            return {k: _walk(v) for k, v in obj.items()}
        return obj

    sanitized = _walk(schema)
    log.info("Schema sanitized. Total vault entries from schema=%d.", len(all_vault))
    return sanitized, all_vault


print("✓ tokenize_pii() defined.")
print("✓ save_vault() defined.")
print("✓ load_vault() defined.")
print("✓ restore_pii() defined.")
print("✓ sanitize_schema() defined.")

✓ tokenize_pii() defined.
✓ save_vault() defined.
✓ load_vault() defined.
✓ restore_pii() defined.
✓ sanitize_schema() defined.


---

## Section 9 — Resume Embedding Stage

### `embed_text(text, model_name) → np.ndarray`

Generates dense vector embeddings using HuggingFace `sentence-transformers`:

- **Primary output:** Full-document embedding for the entire sanitized resume.
- **Optional output:** Chunk-level embeddings (controlled by `CONFIG["embed_chunks"]`).
- Model: `BAAI/bge-base-en-v1.5` — 768-dimensional, English optimized, MTEB top performer.
- BGE models require a query prefix for retrieval tasks; for document embedding, no prefix is added here.

> **Important:** This notebook does **NOT** query ChromaDB, perform Top-K retrieval, or produce `chroma_results.json`. Those operations belong to File 2.

In [10]:
# ────────────────────────────────────────────────────────────────
# RESUME EMBEDDING STAGE
# Uses BAAI/bge-base-en-v1.5 via sentence-transformers.
# Primary output: full-document embedding.
# Optional: chunked embeddings (CONFIG["embed_chunks"]).
# ────────────────────────────────────────────────────────────────

from typing import Optional

# ── Text chunker ─────────────────────────────────────────────────

def _chunk_text(
    text: str,
    chunk_size: int = 300,
    overlap: int = 50,
) -> list[str]:
    """
    Split text into overlapping word-level chunks.

    Parameters
    ----------
    text       : str  — Text to split.
    chunk_size : int  — Maximum words per chunk.
    overlap    : int  — Number of words shared between consecutive chunks.

    Returns
    -------
    list[str]  — List of text chunks.
    """
    if not text.strip():
        return []
    words = text.split()
    chunks: list[str] = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i: i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    log.debug("Text chunked into %d chunks (size=%d, overlap=%d).", len(chunks), chunk_size, overlap)
    return chunks


# ── Model loader (cached) ─────────────────────────────────────────

_LOADED_MODELS: dict[str, Any] = {}

def _load_model(model_name: str) -> Any:
    """Load and cache a SentenceTransformer model."""
    if not _ST_AVAILABLE:
        raise ImportError(
            "sentence-transformers is not installed. "
            "Run: pip install sentence-transformers"
        )
    if model_name not in _LOADED_MODELS:
        log.info("Loading embedding model '%s' (first call — may download) …", model_name)
        _LOADED_MODELS[model_name] = SentenceTransformer(model_name)
        log.info("Model '%s' loaded. Embedding dim=%d.",
                 model_name,
                 _LOADED_MODELS[model_name].get_sentence_embedding_dimension())
    return _LOADED_MODELS[model_name]


# ── embed_text ────────────────────────────────────────────────────

def embed_text(
    text: str,
    model_name: str | None = None,
    normalize: bool = True,
) -> np.ndarray:
    """
    Embed the full sanitized resume text into a single dense vector.

    Parameters
    ----------
    text       : str   — Sanitized resume text.
    model_name : str   — HuggingFace model name (defaults to CONFIG["embedding_model"]).
    normalize  : bool  — L2-normalize the output vector (recommended for cosine similarity).

    Returns
    -------
    np.ndarray  — Shape (embedding_dim,), dtype float32.

    Notes
    -----
    - BGE models benefit from mean-pooling over the full document.
    - For very long documents the model will truncate at its max_seq_length;
      this is acceptable for a full-document representation. Use chunk embeddings
      for fine-grained retrieval (handled by File 2).
    - This function does NOT write to ChromaDB.
    """
    if not text.strip():
        raise ValueError("Cannot embed empty text.")

    name = model_name or CONFIG["embedding_model"]
    model = _load_model(name)

    log.info("Embedding full document (%d chars) …", len(text))
    embedding = model.encode(
        text,
        normalize_embeddings=normalize,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    vec = np.array(embedding, dtype=np.float32)
    log.info("Full-document embedding computed. Shape=%s, dtype=%s.", vec.shape, vec.dtype)
    return vec


# ── embed_chunks ──────────────────────────────────────────────────

def embed_chunks(
    text: str,
    model_name: str | None = None,
    chunk_size: int | None = None,
    chunk_overlap: int | None = None,
    normalize: bool = True,
) -> tuple[np.ndarray, list[str]]:
    """
    Embed the text as overlapping word-level chunks.

    Returns
    -------
    embeddings : np.ndarray   — Shape (n_chunks, embedding_dim), float32.
    chunks     : list[str]    — The text of each corresponding chunk.
    """
    name     = model_name    or CONFIG["embedding_model"]
    c_size   = chunk_size    or CONFIG["chunk_size"]
    c_overlap= chunk_overlap or CONFIG["chunk_overlap"]
    model    = _load_model(name)

    chunks = _chunk_text(text, chunk_size=c_size, overlap=c_overlap)
    if not chunks:
        raise ValueError("Text produced zero chunks.")

    log.info("Embedding %d chunks with model '%s' …", len(chunks), name)
    embeddings = model.encode(
        chunks,
        normalize_embeddings=normalize,
        show_progress_bar=True,
        convert_to_numpy=True,
        batch_size=32,
    )
    arr = np.array(embeddings, dtype=np.float32)
    log.info("Chunk embeddings computed. Shape=%s.", arr.shape)
    return arr, chunks


print("✓ embed_text() defined.")
print("✓ embed_chunks() defined.")

✓ embed_text() defined.
✓ embed_chunks() defined.


---

## Section 10 — Save Outputs

### `save_outputs(structured_schema, embeddings, vault_entries, config)`

Persists all pipeline artifacts to disk:

| File | Content |
|---|---|
| `structured_resume.json` | Sanitized structured schema (all PII replaced by tokens) |
| `pii_vault.json` | Token ↔ original PII mapping (protect with `chmod 600`) |
| `resume_embeddings.npy` | Full-document embedding vector (768-dim float32) |
| `resume_chunks_embeddings.npy` | *(optional)* Per-chunk embeddings array |

In [11]:
# ────────────────────────────────────────────────────────────────
# SAVE OUTPUTS
# ────────────────────────────────────────────────────────────────

def save_outputs(
    structured_schema:   dict,
    full_embeddings:     np.ndarray,
    vault_entries:       list[dict],
    chunk_embeddings:    Optional[np.ndarray] = None,
    config:              dict | None = None,
) -> dict[str, str]:
    """
    Persist all File 1 pipeline artifacts to disk.

    Parameters
    ----------
    structured_schema  : dict         — Sanitized structured JSON schema.
    full_embeddings    : np.ndarray   — Full-document embedding vector.
    vault_entries      : list[dict]   — PII token→original vault entries.
    chunk_embeddings   : np.ndarray   — (optional) Per-chunk embeddings.
    config             : dict         — Pipeline config (defaults to CONFIG global).

    Returns
    -------
    dict[str, str]  — Mapping of artifact name → saved file path.
    """
    cfg = config or CONFIG
    out_cfg = cfg.get("output", {})

    saved: dict[str, str] = {}

    # ── structured_resume.json ────────────────────────────────────
    schema_path = Path(out_cfg.get("structured_resume", "structured_resume.json"))
    schema_path.parent.mkdir(parents=True, exist_ok=True)
    with open(schema_path, "w", encoding="utf-8") as f:
        json.dump(structured_schema, f, indent=2, ensure_ascii=False)
    saved["structured_resume"] = str(schema_path)
    log.info("Structured schema saved → %s", schema_path)

    # ── pii_vault.json ────────────────────────────────────────────
    vault_path = Path(out_cfg.get("pii_vault", "pii_vault.json"))
    save_vault(vault_entries, vault_path)
    saved["pii_vault"] = str(vault_path)

    # ── resume_embeddings.npy ─────────────────────────────────────
    emb_path = Path(out_cfg.get("embeddings", "resume_embeddings.npy"))
    emb_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(str(emb_path), full_embeddings)
    saved["embeddings"] = str(emb_path)
    log.info(
        "Full-document embeddings saved → %s  shape=%s", emb_path, full_embeddings.shape
    )

    # ── resume_chunks_embeddings.npy (optional) ───────────────────
    if chunk_embeddings is not None and cfg.get("embed_chunks", False):
        chunks_path = Path(out_cfg.get("chunks_embeddings", "resume_chunks_embeddings.npy"))
        chunks_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(str(chunks_path), chunk_embeddings)
        saved["chunks_embeddings"] = str(chunks_path)
        log.info(
            "Chunk embeddings saved → %s  shape=%s", chunks_path, chunk_embeddings.shape
        )

    log.info("All outputs saved. Summary: %s", saved)
    return saved


print("✓ save_outputs() defined.")

✓ save_outputs() defined.


---

## Section 11 — Full Pipeline Runner

### `run_pipeline(resume_path, job_description, config, salt)`

Orchestrates all File 1 stages in order:

```
extract_text → parse_structured_schema → detect_pii → tokenize_pii
→ sanitize_schema → embed_text → [embed_chunks] → save_outputs
```

Returns a result dict with references to all artifacts and summary statistics.

In [12]:
# ────────────────────────────────────────────────────────────────
# FULL PIPELINE RUNNER
# ────────────────────────────────────────────────────────────────

def run_pipeline(
    resume_path:     str | Path,
    job_description: str = "",
    config:          dict | None = None,
    salt:            str | None = None,
) -> dict:
    """
    Execute the complete File 1 pipeline.

    Stages (in order):
        1. Text extraction (PDF or DOCX)
        2. Structured schema parsing
        3. PII detection on raw text
        4. PII tokenization of raw text
        5. Schema sanitization (walk + tokenize schema fields)
        6. Add job_description to output (not embedded here)
        7. Full-document embedding on sanitized text
        8. Optional chunk-level embeddings
        9. Save all outputs to disk

    Parameters
    ----------
    resume_path     : str | Path  — Path to resume file (.pdf or .docx).
    job_description : str         — Job description plain text (stored; not embedded here).
    config          : dict        — Pipeline config (defaults to CONFIG global).
    salt            : str         — Hashing salt (defaults to SALT global from Section 4).

    Returns
    -------
    dict with keys:
        raw_text           : str
        sanitized_text     : str
        structured_schema  : dict (sanitized)
        vault_entries      : list[dict]
        full_embedding     : np.ndarray
        chunk_embeddings   : np.ndarray | None
        saved_files        : dict[str, str]
        stats              : dict
    """
    import time
    cfg = config or CONFIG
    s   = salt   or SALT

    log.info("=" * 60)
    log.info("FILE 1 PIPELINE START")
    log.info("=" * 60)
    t0 = time.time()

    # ── Stage 1: Extract text ────────────────────────────────────
    log.info("[Stage 1/8] Extracting text …")
    raw_text = extract_text(resume_path)
    log.info("Stage 1 done. chars=%d", len(raw_text))

    # ── Stage 2: Parse structured schema ─────────────────────────
    log.info("[Stage 2/8] Parsing structured schema …")
    schema = parse_structured_schema(raw_text)
    log.info("Stage 2 done.")

    # ── Stage 3: Detect PII in raw text ──────────────────────────
    log.info("[Stage 3/8] Detecting PII …")
    detections = detect_pii(raw_text, cfg, s)
    log.info("Stage 3 done. detections=%d", len(detections))

    # ── Stage 4: Tokenize raw text ────────────────────────────────
    log.info("[Stage 4/8] Tokenizing PII in raw text …")
    sanitized_text, vault_from_text = tokenize_pii(raw_text, detections, s)
    log.info("Stage 4 done. vault_entries_from_text=%d", len(vault_from_text))

    # ── Stage 5: Sanitize structured schema ──────────────────────
    log.info("[Stage 5/8] Sanitizing structured schema …")
    sanitized_schema, vault_from_schema = sanitize_schema(schema, s)

    # Merge vault entries (deduplicate by token)
    seen_tokens = {e["token"] for e in vault_from_text}
    combined_vault = list(vault_from_text)
    for entry in vault_from_schema:
        if entry["token"] not in seen_tokens:
            combined_vault.append(entry)
            seen_tokens.add(entry["token"])

    # Attach job description (sanitized, not embedded here)
    jd_sanitized = ""
    if job_description.strip():
        jd_dets = detect_pii(job_description, cfg, s)
        jd_sanitized, jd_vault = tokenize_pii(job_description, jd_dets, s)
        for entry in jd_vault:
            if entry["token"] not in seen_tokens:
                combined_vault.append(entry)
                seen_tokens.add(entry["token"])

    sanitized_schema["_job_description_sanitized"] = jd_sanitized
    log.info("Stage 5 done. total_vault_entries=%d", len(combined_vault))

    # ── Stage 6: Full-document embedding ─────────────────────────
    log.info("[Stage 6/8] Embedding full document …")
    full_emb = embed_text(sanitized_text, model_name=cfg["embedding_model"])
    log.info("Stage 6 done. embedding_shape=%s", full_emb.shape)

    # ── Stage 7: Optional chunk embeddings ───────────────────────
    chunk_emb: Optional[np.ndarray] = None
    chunk_texts: list[str] = []
    if cfg.get("embed_chunks", False):
        log.info("[Stage 7/8] Embedding chunks …")
        chunk_emb, chunk_texts = embed_chunks(
            sanitized_text,
            model_name    = cfg["embedding_model"],
            chunk_size    = cfg["chunk_size"],
            chunk_overlap = cfg["chunk_overlap"],
        )
        sanitized_schema["_chunk_texts"] = chunk_texts
        log.info("Stage 7 done. chunks=%d.", len(chunk_texts))
    else:
        log.info("[Stage 7/8] Chunk embedding skipped (embed_chunks=False).")

    # ── Stage 8: Save outputs ─────────────────────────────────────
    log.info("[Stage 8/8] Saving outputs …")
    saved = save_outputs(
        structured_schema = sanitized_schema,
        full_embeddings   = full_emb,
        vault_entries     = combined_vault,
        chunk_embeddings  = chunk_emb,
        config            = cfg,
    )
    log.info("Stage 8 done.")

    elapsed = time.time() - t0
    stats = {
        "raw_text_chars":    len(raw_text),
        "sanitized_chars":   len(sanitized_text),
        "pii_detections":    len(detections),
        "vault_entries":     len(combined_vault),
        "embedding_dim":     int(full_emb.shape[0]),
        "chunks_produced":   len(chunk_texts),
        "elapsed_seconds":   round(elapsed, 2),
    }

    log.info("=" * 60)
    log.info("FILE 1 PIPELINE COMPLETE in %.2fs", elapsed)
    log.info("Stats: %s", stats)
    log.info("Saved: %s", saved)
    log.info("=" * 60)

    return {
        "raw_text":          raw_text,
        "sanitized_text":    sanitized_text,
        "structured_schema": sanitized_schema,
        "vault_entries":     combined_vault,
        "full_embedding":    full_emb,
        "chunk_embeddings":  chunk_emb,
        "saved_files":       saved,
        "stats":             stats,
    }


print("✓ run_pipeline() defined.")

✓ run_pipeline() defined.


---

## Section 12 — Unit Tests (Sample Validation)

These tests use **synthetic data only** — no real PII. They validate each pipeline function in isolation.

Tests:
1. `test_clean_text` — whitespace normalization
2. `test_detect_pii` — detection coverage across all PII categories
3. `test_tokenize_pii` — determinism, token format, no-collision
4. `test_vault_roundtrip` — save/load vault consistency
5. `test_sanitize_schema` — structured schema walk
6. `test_chunk_text` — chunking with overlap
7. `test_embed_text` — embedding shape and dtype

In [ ]:
# ────────────────────────────────────────────────────────────────
# UNIT TESTS — SYNTHETIC DATA ONLY (no real PII)
# ────────────────────────────────────────────────────────────────

import tempfile, traceback

_TEST_SALT = "test_salt_abc123"

_SAMPLE_TEXT = """
ALICE TESTINGTON

alice.testington@fakecorp.io  |  +1 (555) 867-5309  |  github.com/alice-test

123 Maple Street, Springfield  |  ZIP: 90210

SUMMARY
Experienced software engineer with 7 years building distributed systems.

EDUCATION
B.Sc. Computer Science — Fictional University, 2017

SKILLS
Technical: Python, Go, Rust, Kubernetes
Tools: Docker, Terraform, GitHub Actions
Soft Skills: Communication, Mentoring

EXPERIENCE
Senior Engineer | BuildCo | 2020 – Present
• Led migration of monolith to microservices
• Reduced latency by 40% via caching layer

Software Engineer | TechStuff Inc. | 2017 – 2020
• Built RESTful APIs serving 1M RPS
• Improved test coverage from 30% to 90%

PROJECTS
Distributed Task Queue
  Technologies: Python, Redis, Celery
  https://github.com/alice-test/taskqueue

PROFILES
GitHub:   https://github.com/alice-test
LinkedIn: https://www.linkedin.com/in/alice-testington
"""

_SAMPLE_JD = """
We are looking for a Senior Software Engineer to join our platform team.
Required skills: Python, Kubernetes, distributed systems.
"""

_TEST_RESULTS: dict[str, bool] = {}

def _assert(condition: bool, test_name: str, msg: str = "") -> None:
    if condition:
        _TEST_RESULTS[test_name] = True
        print(f"  ✓ {test_name}")
    else:
        _TEST_RESULTS[test_name] = False
        print(f"  ✗ {test_name}  ← FAILED  {msg}")


# ── Test 1: clean_text ────────────────────────────────────────────
def test_clean_text():
    raw = "Hello\r\n  World  \n\n\n\n\nFoo"
    cleaned = _clean_text(raw)
    _assert("\n\n\n" not in cleaned,      "clean_text/excess_blanks")
    _assert("World" in cleaned,           "clean_text/content_preserved")
    _assert(cleaned == cleaned.strip(),   "clean_text/strip_edges")


# ── Test 2: detect_pii ────────────────────────────────────────────
def test_detect_pii():
    dets = detect_pii(_SAMPLE_TEXT, CONFIG, _TEST_SALT)
    types = {d["type"] for d in dets}
    _assert(len(dets) > 0,               "detect_pii/non_empty")
    _assert("email"   in types,          "detect_pii/email_found")
    _assert("phone"   in types,          "detect_pii/phone_found")
    # Check that github profile is correctly typed
    gh_dets = [d for d in dets if d["type"] == "github"]
    _assert(len(gh_dets) > 0,            "detect_pii/github_profile_found")
    # Obfuscated email detection
    obfuscated = "Contact: bob [at] example [dot] com"
    obs_dets = detect_pii(obfuscated, CONFIG, _TEST_SALT)
    obs_types = [d["type"] for d in obs_dets]
    _assert("email" in obs_types,        "detect_pii/obfuscated_email")
    # Social handle detection
    handle_text = "Follow me at @alice_test on Twitter!"
    h_dets = detect_pii(handle_text, CONFIG, _TEST_SALT)
    h_types = {d["type"] for d in h_dets}
    _assert("social_handles" in h_types, "detect_pii/handle_found")


# ── Test 3: tokenize_pii (determinism + format) ───────────────────
def test_tokenize_pii():
    dets = detect_pii(_SAMPLE_TEXT, CONFIG, _TEST_SALT)
    sanitized1, vault1 = tokenize_pii(_SAMPLE_TEXT, dets, _TEST_SALT)
    sanitized2, vault2 = tokenize_pii(_SAMPLE_TEXT, dets, _TEST_SALT)

    _assert(sanitized1 == sanitized2,    "tokenize_pii/deterministic")
    _assert(len(vault1) > 0,             "tokenize_pii/vault_non_empty")
    _assert("alice.testington" not in sanitized1, "tokenize_pii/email_replaced")

    # Token format: <PREFIX_N_hex16>
    for entry in vault1:
        t = entry["token"]
        _assert(
            re.match(r'^<[A-Z]+_\d+_[0-9a-f]{16}>$', t) is not None,
            f"tokenize_pii/token_format[{entry['type']}]",
            f"bad token: {t}",
        )
        break  # check just one example

    # Same value → same token across two calls
    tokens_1 = {e["canonical"]: e["token"] for e in vault1}
    tokens_2 = {e["canonical"]: e["token"] for e in vault2}
    _assert(tokens_1 == tokens_2,        "tokenize_pii/consistent_tokens")


# ── Test 4: vault roundtrip ───────────────────────────────────────
def test_vault_roundtrip():
    dets = detect_pii(_SAMPLE_TEXT, CONFIG, _TEST_SALT)
    _, vault = tokenize_pii(_SAMPLE_TEXT, dets, _TEST_SALT)
    with tempfile.NamedTemporaryFile(suffix=".json", mode="w", delete=False) as tf:
        tf_path = tf.name
    save_vault(vault, tf_path)
    loaded = load_vault(tf_path)
    _assert(
        len(loaded) == len(vault),       "vault_roundtrip/entry_count",
        f"saved={len(vault)} loaded={len(loaded)}",
    )
    for entry in vault:
        _assert(
            entry["token"] in loaded and loaded[entry["token"]] == entry["original"],
            f"vault_roundtrip/mapping[{entry['type']}]",
        )
        break  # check one representative entry
    import os; os.unlink(tf_path)


# ── Test 5: sanitize_schema ───────────────────────────────────────
def test_sanitize_schema():
    schema = parse_structured_schema(_SAMPLE_TEXT)
    s_schema, vault = sanitize_schema(schema, _TEST_SALT)
    # Original email should not appear in string form in the sanitized schema
    schema_str = json.dumps(s_schema)
    _assert(
        "alice.testington@fakecorp.io" not in schema_str,
        "sanitize_schema/email_removed_from_schema",
    )
    _assert(isinstance(s_schema, dict),  "sanitize_schema/returns_dict")


# ── Test 6: chunk_text ────────────────────────────────────────────
def test_chunk_text():
    long_text = " ".join([f"word{i}" for i in range(1000)])
    chunks = _chunk_text(long_text, chunk_size=100, overlap=20)
    _assert(len(chunks) > 1,             "chunk_text/multiple_chunks")
    for c in chunks:
        _assert(len(c.split()) <= 100,   "chunk_text/size_respected")
    # Overlap: last words of chunk N should appear at start of chunk N+1
    if len(chunks) >= 2:
        last_words  = set(chunks[0].split()[-20:])
        first_words = set(chunks[1].split()[:20])
        _assert(bool(last_words & first_words), "chunk_text/overlap_present")


# ── Test 7: embed_text ────────────────────────────────────────────
def test_embed_text():
    if not _ST_AVAILABLE:
        print("  ⚠ test_embed_text SKIPPED — sentence-transformers not installed.")
        return
    sample = "This is a synthetic resume text for testing embedding generation."
    dets = detect_pii(sample, CONFIG, _TEST_SALT)
    sanitized, _ = tokenize_pii(sample, dets, _TEST_SALT)
    vec = embed_text(sanitized, model_name=CONFIG["embedding_model"])
    _assert(isinstance(vec, np.ndarray),   "embed_text/returns_ndarray")
    _assert(vec.ndim == 1,                 "embed_text/1d_vector")
    _assert(vec.dtype == np.float32,       "embed_text/float32_dtype")
    _assert(vec.shape[0] == 768,           "embed_text/dim_768",
            f"actual dim={vec.shape[0]}")
    # L2 norm should be ~1.0 (normalized)
    import math
    norm = float(np.linalg.norm(vec))
    _assert(abs(norm - 1.0) < 0.01,       "embed_text/l2_normalized",
            f"norm={norm:.4f}")


# ── Run all tests ─────────────────────────────────────────────────
print("=" * 55)
print("RUNNING UNIT TESTS")
print("=" * 55)

for test_fn in [
    test_clean_text,
    test_detect_pii,
    test_tokenize_pii,
    test_vault_roundtrip,
    test_sanitize_schema,
    test_chunk_text,
    test_embed_text,
]:
    print(f"\n[ {test_fn.__name__} ]")
    try:
        test_fn()
    except Exception:
        print(f"  ✗ {test_fn.__name__} raised an exception:")
        traceback.print_exc()

passed = sum(1 for v in _TEST_RESULTS.values() if v)
total  = len(_TEST_RESULTS)
print(f"\n{'='*55}")
print(f"Results: {passed}/{total} assertions passed.")
print(f"{'='*55}")

RUNNING UNIT TESTS

[ test_clean_text ]
  ✗ clean_text/excess_blanks  ← FAILED  
  ✓ clean_text/content_preserved
  ✓ clean_text/strip_edges

[ test_detect_pii ]
21:36:38 [INFO    ] resume_pipeline — PII detection complete. 19 raw detections → 12 after dedup.
21:36:38 [DEBUG   ] resume_pipeline —   [email] al***io
21:36:38 [DEBUG   ] resume_pipeline —   [phone] +1***09
21:36:38 [DEBUG   ] resume_pipeline —   [github] gi***st
21:36:38 [DEBUG   ] resume_pipeline —   [address] 12***et
21:36:38 [DEBUG   ] resume_pipeline —   [phone] 90***10
21:36:38 [DEBUG   ] resume_pipeline —   [address] 20***17
21:36:38 [DEBUG   ] resume_pipeline —   [address] 20***20
21:36:38 [DEBUG   ] resume_pipeline —   [address] 20***17
21:36:38 [DEBUG   ] resume_pipeline —   [address] 20***20
21:36:38 [DEBUG   ] resume_pipeline —   [github] ht***ue
21:36:38 [DEBUG   ] resume_pipeline —   [github] ht***st
21:36:38 [DEBUG   ] resume_pipeline —   [linkedin] ht***on
  ✓ detect_pii/non_empty
  ✓ detect_pii/email_found


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 805.52it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


21:36:51 [INFO    ] resume_pipeline — Model 'BAAI/bge-base-en-v1.5' loaded. Embedding dim=768.
21:36:51 [INFO    ] resume_pipeline — Embedding full document (65 chars) …
21:36:51 [INFO    ] resume_pipeline — Full-document embedding computed. Shape=(768,), dtype=float32.
  ✓ embed_text/returns_ndarray
  ✓ embed_text/1d_vector
  ✓ embed_text/float32_dtype
  ✓ embed_text/dim_768
  ✓ embed_text/l2_normalized

Results: 25/26 assertions passed.


---

## Section 13 — Live Pipeline Execution

Set `RESUME_FILE` to the path of your actual resume file (.pdf or .docx).  
Optionally set `JOB_DESCRIPTION` to the job description text.

The cell will run the full pipeline and print a summary.  
All outputs are saved to the paths defined in `CONFIG["output"]`.

In [15]:
# ────────────────────────────────────────────────────────────────
# LIVE PIPELINE EXECUTION
# Set RESUME_FILE and JOB_DESCRIPTION before running.
# ────────────────────────────────────────────────────────────────

# ── User configuration ────────────────────────────────────────────
RESUME_FILE     = "Eshwarnath.pdf"         # ← Change to your resume file path
JOB_DESCRIPTION = "job_des.txt"                   # ← Paste or load the job description here

# ── Optional: load job description from file ──────────────────────
JD_FILE = "job_des.txt"
if not JOB_DESCRIPTION and Path(JD_FILE).exists():
    JOB_DESCRIPTION = Path(JD_FILE).read_text(encoding="utf-8")
    log.info("Loaded job description from %s (%d chars).", JD_FILE, len(JOB_DESCRIPTION))

# ── Run ───────────────────────────────────────────────────────────
if not Path(RESUME_FILE).exists():
    print(f"⚠  Resume file not found: '{RESUME_FILE}'")
    print("   Update RESUME_FILE to point to your actual resume (.pdf or .docx).")
    print("   Skipping live pipeline execution.")
else:
    print("Starting File 1 pipeline …\n")
    result = run_pipeline(
        resume_path     = RESUME_FILE,
        job_description = JOB_DESCRIPTION,
        config          = CONFIG,
        salt            = SALT,
    )

    stats = result["stats"]
    saved = result["saved_files"]

    print("\n" + "="*55)
    print("PIPELINE COMPLETE — SUMMARY")
    print("="*55)
    print(f"  Raw text length      : {stats['raw_text_chars']:,} chars")
    print(f"  Sanitized length     : {stats['sanitized_chars']:,} chars")
    print(f"  PII detections       : {stats['pii_detections']}")
    print(f"  Vault entries        : {stats['vault_entries']}")
    print(f"  Embedding dimension  : {stats['embedding_dim']}")
    print(f"  Chunks produced      : {stats['chunks_produced']}")
    print(f"  Elapsed              : {stats['elapsed_seconds']}s")
    print("\n  Saved files:")
    for name, path in saved.items():
        print(f"    {name:<22} → {path}")
    print("="*55)
    print("\n✓ File 1 complete. Pass the output files to File 2 for retrieval and generation.")

Starting File 1 pipeline …

21:38:02 [INFO    ] resume_pipeline — ============================================================
21:38:02 [INFO    ] resume_pipeline — FILE 1 PIPELINE START
21:38:02 [INFO    ] resume_pipeline — ============================================================
21:38:02 [INFO    ] resume_pipeline — [Stage 1/8] Extracting text …
21:38:02 [DEBUG   ] resume_pipeline — Extracting text from: Eshwarnath.pdf (format=.pdf)
21:38:02 [DEBUG   ] resume_pipeline — PDF extracted via PyMuPDF. Raw chars=2368
21:38:02 [INFO    ] resume_pipeline — Text extraction complete. Cleaned length=2367 chars.
21:38:02 [INFO    ] resume_pipeline — Stage 1 done. chars=2367
21:38:02 [INFO    ] resume_pipeline — [Stage 2/8] Parsing structured schema …
21:38:02 [INFO    ] resume_pipeline — Parsing structured schema …
21:38:02 [DEBUG   ] resume_pipeline — Detected sections: ['personal_info', 'summary', 'education', 'projects', 'languages', 'skills', 'certifications']
21:38:02 [INFO    ] resume_

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]

21:38:04 [INFO    ] resume_pipeline — Chunk embeddings computed. Shape=(2, 768).
21:38:04 [INFO    ] resume_pipeline — Stage 7 done. chunks=2.
21:38:04 [INFO    ] resume_pipeline — [Stage 8/8] Saving outputs …
21:38:04 [INFO    ] resume_pipeline — Structured schema saved → structured_resume.json
21:38:04 [INFO    ] resume_pipeline — Vault saved → pii_vault.json  (21 new + 0 existing = 21 total entries).
  ⚠  Vault saved to: pii_vault.json
  ⚠  IMPORTANT: Restrict file permissions immediately:
       Linux/macOS : chmod 600 pii_vault.json
       Windows     : icacls pii_vault.json /inheritance:r /grant:r "%USERNAME%:F"
21:38:04 [INFO    ] resume_pipeline — Full-document embeddings saved → resume_embeddings.npy  shape=(768,)
21:38:04 [INFO    ] resume_pipeline — Chunk embeddings saved → resume_chunks_embeddings.npy  shape=(2, 768)
21:38:04 [INFO    ] resume_pipeline — All outputs saved. Summary: {'structured_resume': 'structured_resume.json', 'pii_vault': 'pii_vault.json', 'embeddings': 

---

## Appendix — Output File Reference

### `structured_resume.json`
Sanitized structured schema. All PII fields replaced by typed tokens. Example:
```json
{
  "personal_info": {
    "name":    "<NAME_1_a3f92b01c2d3e4f5>",
    "email":   "<EMAIL_1_7c4d1288ef01a3b2>",
    "phone":   "<PHONE_1_9b8e2f3c4d5a6e7f>"
  },
  "profiles": [
    { "type": "github",   "token": "<GITHUB_1_88ef01a3f92b01c2>" },
    { "type": "linkedin", "token": "<LINKEDIN_1_4d1288ef01a3b200>" }
  ],
  ...
}
```

### `pii_vault.json`
Token ↔ original mapping. **Protect with strict file permissions.**
```json
[
  {
    "token":     "<EMAIL_1_7c4d1288ef01a3b2>",
    "original":  "alice@example.com",
    "canonical": "alice@example.com",
    "hash":      "7c4d1288ef01a3b2...",
    "type":      "email"
  }
]
```

### `resume_embeddings.npy`
NumPy array, shape `(768,)`, dtype `float32`. L2-normalized dense vector for the full sanitized resume.  
Load with: `np.load("resume_embeddings.npy")`

### `resume_chunks_embeddings.npy` *(optional)*
NumPy array, shape `(n_chunks, 768)`, dtype `float32`.  
Chunk texts stored under `structured_resume.json["_chunk_texts"]`.

---

## File 2 Handoff

File 2 will consume:
- `structured_resume.json` → context for generation
- `pii_vault.json` → token restoration after generation
- `resume_embeddings.npy` → similarity search against ChromaDB job postings
- `resume_chunks_embeddings.npy` → fine-grained chunk retrieval

File 2 responsibilities (NOT in this notebook):
- ChromaDB ingestion and Top-K retrieval
- Job description embedding and similarity scoring
- ATS keyword analysis
- LLM-based resume generation
- Critic loop and self-refinement